# Graph Neural Networks from Scratch: Message Passing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/graph_neural_networks_from_scratch.ipynb)

Build a graph convolutional network (GCN) from scratch in NumPy: message passing
and backpropagation written out by hand. Train it to near-perfect accuracy on a
friendly graph, then watch the same code collapse on a fraud graph, and learn why
through three ideas every practitioner should know: **homophily**,
**over-smoothing**, and the trap of the aggregate metric.

**Blog post:** [sesen.ai/blog/graph-neural-networks-from-scratch](https://sesen.ai/blog/graph-neural-networks-from-scratch)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

TEAL, AMBER, NAVY, GREY = "#3D9B8F", "#D4A24C", "#1B2D3D", "#9AA7B0"
CLASS_COLOURS = [TEAL, AMBER]
np.random.seed(0)

## Part 1: A graph is just two matrices

To a GCN, a graph is a pair `(X, A)`. The feature matrix `X` is `N x F` (one row
of features per node). The adjacency matrix `A` is `N x N` with `A[i, j] = 1` when
an edge connects nodes `i` and `j`.

We generate graphs with a **stochastic block model** and a *homophily knob*: the
fraction of each node's edges that land on the same class. `homophily=0.9` is a
citation graph; `homophily=0.1` is a fraud graph where bad actors wire themselves
to legitimate accounts. Node features are class-mean + noise, so a model that
ignores the graph still has signal: the graph either denoises it (homophily) or
corrupts it (heterophily).

In [ ]:
def make_graph(n_per_class=150, n_classes=2, homophily=0.8, avg_degree=10,
               feat_dim=20, feat_sep=0.9, noise=1.0, seed=0):
    rng = np.random.default_rng(seed)
    n = n_per_class * n_classes
    y = np.repeat(np.arange(n_classes), n_per_class)
    # edge probabilities chosen so expected degree = avg_degree and the
    # within-class edge fraction equals `homophily`
    m_edges = n * avg_degree / 2.0
    same_pairs = n_classes * n_per_class * (n_per_class - 1) / 2.0
    diff_pairs = n * n / 2.0 - same_pairs
    p_in = homophily * m_edges / same_pairs
    p_out = (1.0 - homophily) * m_edges / diff_pairs
    same = y[:, None] == y[None, :]
    probs = np.where(same, p_in, p_out)
    upper = np.triu(rng.random((n, n)) < probs, k=1)
    A = (upper | upper.T).astype(float)
    means = rng.normal(0, feat_sep, size=(n_classes, feat_dim))
    X = means[y] + rng.normal(0, noise, size=(n, feat_dim))
    return X.astype(float), A, y


def edge_homophily(A, y):
    iu = np.triu_indices_from(A, k=1)
    edges = A[iu] > 0
    same = y[iu[0]] == y[iu[1]]
    return float((same & edges).sum() / edges.sum())

Let us look at a homophilic graph next to a heterophilic one. Node positions come from the graph Laplacian's eigenvectors (a spectral layout, no extra libraries needed).

In [ ]:
def spectral_layout(A, seed=0):
    L = np.diag(A.sum(1)) - A
    _, vecs = eigh(L)
    coords = vecs[:, 1:3]
    rng = np.random.default_rng(seed)
    return coords + rng.normal(0, coords.std() * 0.08, coords.shape)


def draw_graph(ax, A, y, coords, title):
    iu = np.triu_indices_from(A, k=1)
    for a, b in zip(iu[0][A[iu] > 0], iu[1][A[iu] > 0]):
        ax.plot(coords[[a, b], 0], coords[[a, b], 1], color=GREY, lw=0.25, alpha=0.3)
    for c in np.unique(y):
        m = y == c
        ax.scatter(coords[m, 0], coords[m, 1], s=22, c=CLASS_COLOURS[c],
                   edgecolors="white", linewidths=0.4, label=f"class {c}")
    ax.set_title(title, fontweight="bold"); ax.set_xticks([]); ax.set_yticks([])


fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, h, t in [(axes[0], 0.92, "Homophilic (h=0.92)"), (axes[1], 0.08, "Heterophilic (h=0.08)")]:
    X, A, y = make_graph(n_per_class=60, homophily=h, avg_degree=8, seed=11)
    draw_graph(ax, A, y, spectral_layout(A, 11), t)
axes[0].legend(frameon=False); plt.tight_layout(); plt.show()

## Part 2: The GCN, message passing by hand

A GCN propagates across the graph using the **symmetric-normalised adjacency**:

$$\hat{A} = \tilde{D}^{-1/2}\,\tilde{A}\,\tilde{D}^{-1/2}, \qquad \tilde{A} = A + I$$

Adding the identity gives each node a self-loop so it keeps its own message; the
degree terms turn the raw neighbour sum into a weighted average. We compute it
once; it never changes during training.

In [ ]:
def normalise_adjacency(A):
    A_tilde = A + np.eye(A.shape[0])      # add self-loops
    deg = A_tilde.sum(1)
    d_inv_sqrt = 1.0 / np.sqrt(deg)
    return d_inv_sqrt[:, None] * A_tilde * d_inv_sqrt[None, :]

The model is two matrix multiplies wrapped around propagation. We write the
forward pass, the hand-derived backward pass, and an Adam training loop with
masked, class-weighted cross-entropy (only labelled nodes contribute to the loss,
but every node still participates in propagation).

In [ ]:
def relu(x): return np.maximum(x, 0.0)
def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)


class GCN:
    def __init__(self, in_dim, hidden, out_dim, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, np.sqrt(2 / (in_dim + hidden)), (in_dim, hidden))
        self.W2 = rng.normal(0, np.sqrt(2 / (hidden + out_dim)), (hidden, out_dim))

    def forward(self, S, X):
        self.SX  = S @ X                       # 1. mix each node with its neighbours
        self.Z1  = self.SX @ self.W1
        self.A1  = relu(self.Z1)               # 2. transform + non-linearity
        self.SA1 = S @ self.A1                 # 3. mix again (now 2 hops out)
        self.Z2  = self.SA1 @ self.W2          # 4. transform to class logits
        return softmax(self.Z2)

    def backward(self, S, P, y, mask, class_w):
        n = P.shape[0]
        onehot = np.zeros_like(P); onehot[np.arange(n), y] = 1.0
        w = class_w[y] * mask
        dZ2 = (P - onehot) * (w[:, None] / w.sum())
        dW2 = self.SA1.T @ dZ2
        dA1 = S @ dZ2 @ self.W2.T              # S symmetric -> S.T = S
        dZ1 = dA1 * (self.Z1 > 0)
        dW1 = self.SX.T @ dZ1
        return dW1, dW2

    def loss(self, P, y, mask, class_w):
        w = class_w[y] * mask
        return float((w * -np.log(P[np.arange(len(y)), y] + 1e-12)).sum() / w.sum())

    def fit(self, S, X, y, mask, epochs=200, lr=0.01, weight_decay=5e-4, class_weight=None):
        c = int(y.max() + 1)
        cw = np.ones(c) if class_weight is None else np.asarray(class_weight, float)
        m1 = m2 = v1 = v2 = 0; b1, b2, eps = 0.9, 0.999, 1e-8
        for t in range(1, epochs + 1):
            P = self.forward(S, X)
            dW1, dW2 = self.backward(S, P, y, mask, cw)
            dW1 += weight_decay * self.W1; dW2 += weight_decay * self.W2
            m1 = b1*m1 + (1-b1)*dW1; v1 = b2*v1 + (1-b2)*dW1*dW1
            m2 = b1*m2 + (1-b1)*dW2; v2 = b2*v2 + (1-b2)*dW2*dW2
            self.W1 -= lr * (m1/(1-b1**t)) / (np.sqrt(v1/(1-b2**t)) + eps)
            self.W2 -= lr * (m2/(1-b1**t)) / (np.sqrt(v2/(1-b2**t)) + eps)
        return self

    def predict(self, S, X): return self.forward(S, X).argmax(1)


def split_mask(y, train_per_class=20, seed=0):
    rng = np.random.default_rng(seed); train = np.zeros(len(y), bool)
    for c in np.unique(y):
        idx = np.where(y == c)[0]; rng.shuffle(idx); train[idx[:train_per_class]] = True
    return train, ~train

### Sanity check: are the hand-derived gradients correct?

Compare the analytic gradient against a finite-difference estimate. A tiny
relative error confirms the backward pass.

In [ ]:
X, A, y = make_graph(n_per_class=20, homophily=0.8, feat_dim=6, seed=1)
S = normalise_adjacency(A); train, _ = split_mask(y, 8, seed=1)
net = GCN(6, 5, 2, seed=2); cw = np.array([1.0, 2.0])
dW1, dW2 = net.backward(S, net.forward(S, X), y, train, cw)
eps, errs = 1e-5, []
for W, dW in [(net.W1, dW1), (net.W2, dW2)]:
    for _ in range(6):
        i, j = np.random.randint(W.shape[0]), np.random.randint(W.shape[1])
        o = W[i, j]
        W[i, j] = o + eps; lp = net.loss(net.forward(S, X), y, train, cw)
        W[i, j] = o - eps; lm = net.loss(net.forward(S, X), y, train, cw)
        W[i, j] = o
        errs.append(abs((lp - lm)/(2*eps) - dW[i, j]) / (abs(dW[i, j]) + 1e-12))
print(f"max relative gradient error: {max(errs):.2e}  (want < 1e-4)")

## Part 3: Train on a homophilic graph

On a graph where 85% of edges connect same-class nodes, a two-layer GCN reaches
near-perfect accuracy from only 20 labelled nodes per class. We plot the learned
node embeddings (the pre-softmax logits): the two classes pull apart.

In [ ]:
X, A, y = make_graph(n_per_class=150, homophily=0.85, seed=12)
S = normalise_adjacency(A); train, test = split_mask(y, 20, seed=12)
net = GCN(X.shape[1], 16, 2, seed=13).fit(S, X, y, train, epochs=200, lr=0.02)
print(f"edge homophily: {edge_homophily(A, y):.2f}")
print(f"test accuracy:  {(net.predict(S, X)[test] == y[test]).mean():.3f}")

net.forward(S, X)
Z = net.Z2  # pre-softmax logits = the learned node embeddings
plt.figure(figsize=(6, 5))
for c in np.unique(y):
    m = y == c
    plt.scatter(Z[m, 0], Z[m, 1], s=14, c=CLASS_COLOURS[c], edgecolors="white",
                linewidths=0.3, label=f"class {c}")
plt.xlabel("logit dim 0"); plt.ylabel("logit dim 1"); plt.legend(frameon=False)
plt.title("Learned node embeddings", fontweight="bold"); plt.show()

## Part 4: The homophily assumption

The accuracy above hid an assumption: **neighbours tend to share your label**.
Averaging neighbours only helps if they look like you. We sweep the full homophily
range and compare the GCN against an MLP that uses the identical architecture but
ignores the graph (set `S = I`). The result is a **U**, not a steady decline.

In [ ]:
def mlp_acc(X, y, train, test, seed=4):
    I = np.eye(len(y))
    m = GCN(X.shape[1], 16, 2, seed=seed).fit(I, X, y, train, epochs=200, lr=0.01)
    return (m.predict(I, X)[test] == y[test]).mean()

hs = [0.95, 0.85, 0.75, 0.65, 0.55, 0.5, 0.45, 0.35, 0.25, 0.15, 0.05]
eh, ga, ma = [], [], []
for h in hs:
    X, A, y = make_graph(n_per_class=150, homophily=h, seed=3)
    S = normalise_adjacency(A); train, test = split_mask(y, 20, seed=3)
    eh.append(edge_homophily(A, y))
    g = GCN(X.shape[1], 16, 2, seed=4).fit(S, X, y, train, epochs=200, lr=0.01)
    ga.append((g.predict(S, X)[test] == y[test]).mean())
    ma.append(mlp_acc(X, y, train, test))

o = np.argsort(eh); eh, ga, ma = np.array(eh)[o], np.array(ga)[o], np.array(ma)[o]
plt.figure(figsize=(8.5, 5))
plt.axvspan(0.4, 0.6, color=AMBER, alpha=0.12)
plt.text(0.5, 0.97, "danger zone\nno neighbour signal", ha="center", va="top",
         color=AMBER, fontweight="bold")
plt.plot(eh, ga, "o-", color=TEAL, lw=2.5, label="GCN (uses the graph)")
plt.plot(eh, ma, "s--", color=NAVY, lw=2, label="MLP (ignores the graph)")
plt.xlabel("edge homophily"); plt.ylabel("test accuracy"); plt.ylim(0.6, 1.04)
plt.legend(frameon=False, loc="lower right"); plt.grid(alpha=0.2); plt.show()

The GCN wins big at high homophily and *also* at very low homophily, where the
graph is consistently heterophilic (a two-layer GCN sees two hops, and the friend
of my enemy is my enemy, so the graph becomes homophilic at two hops). The danger
zone is the **middle**, around homophily 0.5, where a neighbour is a coin flip and
averaging adds noise to features that were fine on their own. This is the Zhu et al.
(2020) insight: low neighbourhood *informativeness*, not heterophily per se, breaks
a GCN.

## Part 5: Over-smoothing

Each propagation step pulls every node toward the average of its neighbours.
Repeat enough and all nodes converge to nearly the same vector. We measure it with
**Dirichlet energy** (how different connected nodes still are): it decays toward
zero. The practical cost is that deep GCNs lose accuracy.

In [ ]:
def deep_gcn_acc(S, X, y, train, test, n_layers, hidden=16, epochs=200, lr=0.01, seed=6):
    rng = np.random.default_rng(seed)
    dims = [X.shape[1]] + [hidden]*(n_layers-1) + [2]
    Ws = [rng.normal(0, np.sqrt(2/(a+b)), (a, b)) for a, b in zip(dims[:-1], dims[1:])]
    m = [0]*len(Ws); v = [0]*len(Ws); b1, b2, eps = 0.9, 0.999, 1e-8
    for t in range(1, epochs+1):
        H = X; cache = []
        for i, W in enumerate(Ws):
            pre = (S @ H) @ W; cache.append((H, pre))
            H = relu(pre) if i < len(Ws)-1 else pre
        P = softmax(H); n = P.shape[0]
        oh = np.zeros_like(P); oh[np.arange(n), y] = 1.0
        dpre = (P - oh) * (train[:, None] / train.sum())
        grads = [None]*len(Ws)
        for i in reversed(range(len(Ws))):
            Hi, _ = cache[i]; grads[i] = (S @ Hi).T @ dpre
            if i > 0:
                dpre = (S @ dpre @ Ws[i].T) * (cache[i-1][1] > 0)
        for i in range(len(Ws)):
            m[i] = b1*m[i] + (1-b1)*grads[i]; v[i] = b2*v[i] + (1-b2)*grads[i]**2
            Ws[i] -= lr * (m[i]/(1-b1**t)) / (np.sqrt(v[i]/(1-b2**t)) + eps)
    H = X
    for i, W in enumerate(Ws):
        H = (S @ H) @ W; H = relu(H) if i < len(Ws)-1 else H
    return (H.argmax(1)[test] == y[test]).mean()


def dirichlet_energy(S, H):
    diff = H[:, None, :] - H[None, :, :]; w = (S > 0).astype(float)
    return float((w[:, :, None] * diff**2).sum() / (2 * S.shape[0]))

X, A, y = make_graph(n_per_class=150, homophily=0.8, feat_sep=0.4, noise=1.2, seed=5)
S = normalise_adjacency(A); train, test = split_mask(y, 15, seed=5)
H = X.copy(); energy = [dirichlet_energy(S, H)]
for _ in range(12):
    H = S @ H; energy.append(dirichlet_energy(S, H))
energy = np.array(energy) / energy[0]
depths = [2, 3, 4, 6, 8, 10]
acc = [deep_gcn_acc(S, X, y, train, test, L) for L in depths]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].semilogy(range(13), energy, "o-", color=TEAL, lw=2.5)
ax[0].set_xlabel("propagation steps k (= layers)"); ax[0].set_ylabel("Dirichlet energy (rel.)")
ax[0].set_title("Message passing collapses embeddings", fontweight="bold"); ax[0].grid(alpha=0.2, which="both")
ax[1].plot(depths, acc, "o-", color=NAVY, lw=2.5); ax[1].axhline(0.5, ls=":", color=AMBER, label="random")
ax[1].set_xlabel("number of GCN layers"); ax[1].set_ylabel("test accuracy"); ax[1].set_ylim(0.4, 1.02)
ax[1].set_title("Deeper is not better", fontweight="bold"); ax[1].legend(frameon=False); ax[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()

Most production GCNs use just two or three layers. To reach further without
over-smoothing, use residual or jumping-knowledge connections that preserve the
earlier, un-smoothed representations.

## Part 6: Why your fraud GNN underperforms

Real fraud data has two properties: it is **imbalanced** (few fraud nodes) and the
fraud is **heterophilic in a way the aggregate hides**. We build such a graph and
print two homophily numbers, then benchmark four models on the fraud class (where
accuracy is useless: predict all-legit and score 92%).

In [ ]:
def make_fraud_graph(seed=7):
    rng = np.random.default_rng(seed)
    n_legit, n_fraud, fd = 1380, 120, 20; n = n_legit + n_fraud
    y = np.array([0]*n_legit + [1]*n_fraud)
    means = rng.normal(0, 0.8, (2, fd)); X = means[y] + rng.normal(0, 1.0, (n, fd))
    p_ll, p_fl, p_ff = 0.012, 0.020, 0.002
    same = y[:, None] == y[None, :]; fpair = (y[:, None] == 1) | (y[None, :] == 1)
    probs = np.where(same & ~fpair, p_ll, 0.0)
    probs = np.where(y[:, None] != y[None, :], p_fl, probs)
    probs = np.where((y[:, None] == 1) & (y[None, :] == 1), p_ff, probs)
    up = np.triu(rng.random((n, n)) < probs, k=1)
    return X.astype(float), (up | up.T).astype(float), y


def fraud_metrics(pred, y, mask):
    yt, yp = y[mask], pred[mask]
    tp = ((yp==1)&(yt==1)).sum(); fp = ((yp==1)&(yt==0)).sum(); fn = ((yp==0)&(yt==1)).sum()
    pr = tp/(tp+fp) if tp+fp else 0; rc = tp/(tp+fn) if tp+fn else 0
    return pr, rc, (2*pr*rc/(pr+rc) if pr+rc else 0)

X, A, y = make_fraud_graph(7); S = normalise_adjacency(A)
train, test = split_mask(y, 40, seed=7)
fraud_purity = np.mean([ (y[np.where(A[i]>0)[0]]==1).mean() for i in np.where(y==1)[0] if A[i].sum() ])
print(f"fraud rate: {y.mean():.3f}   global edge homophily: {edge_homophily(A, y):.2f}")
print(f"fraud-node neighbour purity: {fraud_purity:.2f}  (almost all neighbours are legit!)")

w = [1.0, 12.0]; I = np.eye(len(y)); res = {}
res["MLP (graph-blind)"]      = fraud_metrics(GCN(X.shape[1],16,2,8).fit(I, X, y, train, class_weight=w).predict(I, X), y, test)
res["GCN (vanilla)"]          = fraud_metrics(GCN(X.shape[1],16,2,8).fit(S, X, y, train).predict(S, X), y, test)
res["GCN + class weights"]    = fraud_metrics(GCN(X.shape[1],16,2,8).fit(S, X, y, train, class_weight=w).predict(S, X), y, test)
deg = A.sum(1, keepdims=True); deg[deg==0] = 1
Xcat = np.concatenate([X, (A/deg) @ X], axis=1)   # ego features + neighbour mean, kept separate
res["Ego-separated + weights"] = fraud_metrics(GCN(Xcat.shape[1],16,2,9).fit(I, Xcat, y, train, class_weight=w).predict(I, Xcat), y, test)

print(f"\n{'model':>26} {'prec':>6} {'rec':>6} {'F1':>6}")
for k, (p, r, f) in res.items():
    print(f"{k:>26} {p:>6.2f} {r:>6.2f} {f:>6.2f}")

In [ ]:
labels = list(res); pr = [res[k][0] for k in labels]; rc = [res[k][1] for k in labels]; f1 = [res[k][2] for k in labels]
x = np.arange(len(labels)); width = 0.26
plt.figure(figsize=(9.5, 5))
plt.bar(x-width, pr, width, label="precision", color=GREY)
plt.bar(x, rc, width, label="recall", color=TEAL)
plt.bar(x+width, f1, width, label="F1", color=AMBER)
for i, v in enumerate(f1):
    plt.text(x[i]+width, v+0.02, f"{v:.2f}", ha="center", fontweight="bold")
plt.xticks(x, [l.replace(" ", "\n", 1) for l in labels]); plt.ylim(0, 1.08)
plt.ylabel("fraud-class score"); plt.legend(frameon=False, ncol=3, loc="upper center")
plt.title("Vanilla message passing does worse than ignoring the graph", fontweight="bold")
plt.grid(alpha=0.2, axis="y"); plt.tight_layout(); plt.show()

The global edge homophily reads a healthy 0.77, but fraud-node neighbour
purity is near 0.01: almost every neighbour of a fraud node is legitimate. The
aggregate number, dominated by the 92% legit majority, completely masks the
minority-class structure. The vanilla GCN ends up *worse* than ignoring the graph;
class weights on top of it collapse to predicting all-fraud; only keeping each
node's own features separate from its neighbours (ego-separation) recovers
performance.

**Takeaway for any fraud project:** a GNN is not automatically better than a
tabular model. Always benchmark a graph-blind baseline first.

## Exercises

1. **GAT-style attention.** Replace the fixed averaging weights in `S` with learned
   per-edge weights (a small attention score from concatenated endpoint features).
   Does it help in the danger zone (homophily ~0.5)?
2. **Neighbour sampling (GraphSAGE).** Instead of using the full `A`, sample a fixed
   number of neighbours per node each step. Measure the accuracy/speed trade-off.
3. **Residual connections.** Add `H = H + relu(...)` to the `DeepGCN` and re-run the
   over-smoothing experiment. How deep can you go before accuracy falls?
4. **Real data.** Load the Cora citation graph (e.g. via `torch_geometric` or the
   raw Planetoid files) and confirm the from-scratch GCN reaches ~80% test accuracy.
5. **Imbalance fixes.** On the fraud graph, try focal loss or neighbour-balanced
   sampling instead of flat class weights. Can you beat the ego-separated F1?